In [42]:
from qibo.hamiltonians import SymbolicHamiltonian
from qibo.symbols import X, Y, Z
from qibo.backends import _check_backend
import numpy as np

def z_plus(j):
    return (X(j) + 1j*Y(j)) / 2

def z_minus(j):
    return (X(j) - 1j*Y(j)) / 2

class FidelityWitness:
    def __init__(self, N, boundaries=False, backend=None):
        self.N = N
        self.boundaries = boundaries
        self.backend = _check_backend(backend)

    def get_xxz_folded_hamiltonian(self):
        ham = 0
        if self.boundaries:
            for j in range(self.N-1):
                ham += -(1/8)*(1+Z(j)*Z(j+3))*(X(j+1)*X(j+2)+Y(j+1)*Y(j+2))
        else:
            for j in range(0, self.N-3):
                ham += -(1/8)*(1+Z(j)*Z(j+3))*(X(j+1)*X(j+2)+Y(j+1)*Y(j+2))
            j = 0
            ham += -(1/8)*(1+Z(j+2))*(X(j)*X(j+1)+Y(j)*Y(j+1))
            j = self.N-3
            ham += -(1/8)*(1+Z(j))*(X(j+1)*X(j+2)+Y(j+1)*Y(j+2))
        ham = SymbolicHamiltonian(ham, backend=self.backend)

        return ham

    def get_q1(self):
        q1 = 0
        if self.boundaries:
            for j in range(self.N+2):
                q1 += (1/2)*(1-Z(j))
        else:
            for j in range(0, self.N):
                q1 += (1/2)*(1-Z(j))
        q1 = SymbolicHamiltonian(q1, backend=self.backend)

        return q1

    def get_q2(self):
        q2 = 0
        if self.boundaries:
            for j in range(self.N+1):
                q2 += (1/2)*(1-Z(j)*Z(j+1))
        else:
            for j in range(0, self.N-1):
                q2 += (1/2)*(1-Z(j)*Z(j+1))

            q2 += (1/2)*(1-Z(0))
            q2 += (1/2)*(1-Z(self.N-1))

        q2 = SymbolicHamiltonian(q2, backend=self.backend)

        return q2
    
    def get_q3(self):
        q3 = 0
        if self.boundaries:
            for j in range(self.N-1):
                q3 += 1j*(Z(j)+Z(j+3))*(z_plus(j+1)*z_minus(j+2)-z_minus(j+1)*z_plus(j+2))/4
        else:
            for j in range(0, self.N-3):
                q3 += 1j*(Z(j)+Z(j+3))*(z_plus(j+1)*z_minus(j+2)-z_minus(j+1)*z_plus(j+2))/4

            q3 += 1j*(z_plus(0)*z_minus(1)-z_minus(0)*z_plus(1))/4 + 1j*Z(2)*(z_plus(0)*z_minus(1)-z_minus(0)*z_plus(1))/4
            q3 += 1j*Z(self.N-3)*(z_plus(self.N-2)*z_minus(self.N-1)-z_minus(self.N-2)*z_plus(self.N-1))/4 + 1j*(z_plus(self.N-2)*z_minus(self.N-1)-z_minus(self.N-2)*z_plus(self.N-1))/4

        q3 = SymbolicHamiltonian(q3, backend=self.backend)

        return q3
    
    def get_q4(self):
        q4 = 0
        if self.boundaries:
            for j in range(self.N-1):
                q4 += -(1+Z(j)*Z(j+3))*(z_plus(j+1)*z_minus(j+2)+z_minus(j+1)*z_plus(j+2))/4
        else:
            for j in range(0, self.N-3):
                q4 += -(1+Z(j)*Z(j+3))*(z_plus(j+1)*z_minus(j+2)+z_minus(j+1)*z_plus(j+2))/4

            q4 += -(z_plus(0)*z_minus(1)+z_minus(0)*z_plus(1))/4 -Z(2)*(z_plus(0)*z_minus(1)+z_minus(0)*z_plus(1))/4
            q4 += -Z(self.N-3)*(z_plus(self.N-2)*z_minus(self.N-1)+z_minus(self.N-2)*z_plus(self.N-1))/4 -(z_plus(self.N-2)*z_minus(self.N-1)+z_minus(self.N-2)*z_plus(self.N-1))/4

        q4 = SymbolicHamiltonian(q4, backend=self.backend)

        return q4
    
    def get_q5(self):
        q5 = 0
        if self.boundaries:
            for j in range(self.N-3):
                q5 += (Z(j)+Z(j+4))*(X(j+1)*Y(j+3)-Y(j+1)*X(j+3)) + (Z(j)+Z(j+5))*(X(j+1)*X(j+2)+Y(j+1)*Y(j+2))*(Y(j+3)*X(j+4)-X(j+3)*Y(j+4))
                + (1+Z(j)*Z(j+4))*Z(j+2)*(X(j+1)*Y(j+3)-Y(j+1)*X(j+3))
        else:
            for j in range(0, self.N-5):
                q5 += (Z(j)+Z(j+4))*(X(j+1)*Y(j+3)-Y(j+1)*X(j+3)) + (Z(j)+Z(j+5))*(X(j+1)*X(j+2)+Y(j+1)*Y(j+2))*(Y(j+3)*X(j+4)-X(j+3)*Y(j+4))
                + (1+Z(j)*Z(j+4))*Z(j+2)*(X(j+1)*Y(j+3)-Y(j+1)*X(j+3))

            q5 += (1+Z(3))*(X(0)*Y(2)-Y(0)*X(2)) + (1+Z(4))*(X(0)*X(1)+Y(0)*Y(1))*(Y(2)*X(3)-X(2)*Y(3))
            + (1+Z(3))*Z(1)*(X(0)*Y(2)-Y(0)*X(2))
            q5 += (Z(self.N-5)+Z(self.N-1))*(X(self.N-4)*Y(self.N-2)-Y(self.N-4)*X(self.N-2)) + (Z(self.N-5)+1)*(X(self.N-4)*X(self.N-3)+Y(self.N-4)*Y(self.N-3))*(Y(self.N-2)*X(self.N-1)-X(self.N-2)*Y(self.N-1))
            + (1+Z(self.N-5)*Z(self.N-1))*Z(self.N-3)*(X(self.N-4)*Y(self.N-2)-Y(self.N-4)*X(self.N-2))
               
        q5 = SymbolicHamiltonian(q5, backend=self.backend)

        return q5
    
    def get_witness(self, Q_list, q_target, q_exp):
        import numpy as np
        import cvxpy as cp

        # ... your Q_list and q_target ...

        # Add identity to charges
        #Q_list = [H / np.linalg.norm(H), Q2 / np.linalg.norm(Q2)]
        Q_list = [Q / np.linalg.norm(Q) for Q in Q_list]
        #q_target = np.array([np.vdot(psi_target, Q @ psi_target).real for Q in Q_list])
        I = np.eye(Q_list[0].shape[0])
        Q_list.append(I)
        q_target = np.append(q_target, 1.0)
        q_exp = np.append(q_exp, 1.0)
        
        alpha = cp.Variable(len(Q_list))
        W_expr = sum(alpha[i] * Q_list[i] for i in range(len(Q_list)))

        constraints = [
            W_expr >> 0,
            alpha @ q_target == 1   # normalize witness to have expectation 1 on target
        ]

        objective = cp.Maximize(0)  # can also be any objective; here just maximize fidelity bound

        prob = cp.Problem(objective, constraints)
        prob.solve(solver=cp.SCS, eps=1e-5, verbose=True)

        print("Status:", prob.status)
        print("alpha:", alpha.value)
        print("Fidelity lower bound:", alpha.value @ q_target if alpha.value is not None else None)
        print("Fidelity witness on exp state:", alpha.value @ q_exp if alpha.value is not None else None)



In [43]:
import qibo
qibo.set_backend("numpy")
witness = FidelityWitness(N=6, boundaries=True, backend=None)

[Qibo 0.2.21|INFO|2025-09-17 13:19:08]: Using numpy backend on /CPU:0


In [44]:
ham = witness.get_xxz_folded_hamiltonian().matrix
q1 = witness.get_q1().matrix
q2 = witness.get_q2().matrix
q3 = witness.get_q3().matrix
q4 = witness.get_q4().matrix
q5 = witness.get_q5().matrix

[Qibo 0.2.21|WARNING|2025-09-17 13:19:08]: Calculating the dense form of a symbolic Hamiltonian. This operation is memory inefficient.
[Qibo 0.2.21|WARNING|2025-09-17 13:19:08]: Calculating the dense form of a symbolic Hamiltonian. This operation is memory inefficient.
[Qibo 0.2.21|WARNING|2025-09-17 13:19:08]: Calculating the dense form of a symbolic Hamiltonian. This operation is memory inefficient.
[Qibo 0.2.21|WARNING|2025-09-17 13:19:08]: Calculating the dense form of a symbolic Hamiltonian. This operation is memory inefficient.
[Qibo 0.2.21|WARNING|2025-09-17 13:19:09]: Calculating the dense form of a symbolic Hamiltonian. This operation is memory inefficient.
[Qibo 0.2.21|WARNING|2025-09-17 13:19:09]: Calculating the dense form of a symbolic Hamiltonian. This operation is memory inefficient.


In [45]:
print(np.linalg.norm(q1 @ ham - ham @ q1))
print(np.linalg.norm(q2 @ ham - ham @ q2))
print(np.linalg.norm(q3 @ ham - ham @ q3))
print(np.linalg.norm(q4 @ ham - ham @ q4))
print(np.linalg.norm(q5 @ ham - ham @ q5))

0.0
0.0
4.898979485566356
0.0
69.74238309665078


In [46]:
Q_list = [ham, q1, q2]
q_target = [1,4,8]
q_exp = [0.9,3.5,7.0]
witness.get_witness(Q_list, q_target, q_exp)

(CVXPY) Sep 17 01:19:09 PM: Your problem has 4 variables, 65537 constraints, and 0 parameters.
(CVXPY) Sep 17 01:19:09 PM: It is compliant with the following grammars: DCP, DQCP
(CVXPY) Sep 17 01:19:09 PM: (If you need to solve this problem multiple times, but with different data, consider using parameters.)
(CVXPY) Sep 17 01:19:09 PM: CVXPY will first compile your problem; then, it will invoke a numerical solver to obtain a solution.
(CVXPY) Sep 17 01:19:09 PM: Your problem is compiled with the CPP canonicalization backend.
(CVXPY) Sep 17 01:19:09 PM: Compiling problem (target solver=SCS).
(CVXPY) Sep 17 01:19:09 PM: Reduction chain: Complex2Real -> FlipObjective -> Dcp2Cone -> CvxAttr2Constr -> ConeMatrixStuffing -> SCS
(CVXPY) Sep 17 01:19:09 PM: Applying reduction Complex2Real
(CVXPY) Sep 17 01:19:09 PM: Applying reduction FlipObjective
(CVXPY) Sep 17 01:19:09 PM: Applying reduction Dcp2Cone
(CVXPY) Sep 17 01:19:09 PM: Applying reduction CvxAttr2Constr
(CVXPY) Sep 17 01:19:09 PM: A

(CVXPY) Sep 17 01:19:09 PM: Applying reduction SCS
(CVXPY) Sep 17 01:19:09 PM: Finished problem compilation (took 1.120e-01 seconds).
(CVXPY) Sep 17 01:19:09 PM: Invoking solver SCS  to obtain a solution.


                                     CVXPY                                     
                                     v1.7.2                                    
-------------------------------------------------------------------------------
                                  Compilation                                  
-------------------------------------------------------------------------------
-------------------------------------------------------------------------------
                                Numerical solver                               
-------------------------------------------------------------------------------
------------------------------------------------------------------
	       SCS v3.2.8 - Splitting Conic Solver
	(c) Brendan O'Donoghue, Stanford University, 2012
------------------------------------------------------------------
problem:  variables n: 4, constraints m: 131329
cones: 	  z: primal zero / dual free vars: 1
	  s: psd vars: 131328, ssize: 1
setti

(CVXPY) Sep 17 01:19:10 PM: Problem status: optimal
(CVXPY) Sep 17 01:19:10 PM: Optimal value: 0.000e+00
(CVXPY) Sep 17 01:19:10 PM: Compilation took 1.120e-01 seconds
(CVXPY) Sep 17 01:19:10 PM: Solver (including time spent in interface) took 1.342e+00 seconds


    25| 3.94e-07  1.86e-06  2.32e-07 -1.16e-07  1.00e-01  1.34e+00 
------------------------------------------------------------------
status:  solved
timings: total: 1.34e+00s = setup: 2.43e-02s + solve: 1.32e+00s
	 lin-sys: 2.56e-02s, cones: 1.27e+00s, accel: 3.21e-03s
------------------------------------------------------------------
objective = -0.000000
------------------------------------------------------------------
-------------------------------------------------------------------------------
                                    Summary                                    
-------------------------------------------------------------------------------
Status: optimal
alpha: [ 0.00890711 -0.02047306  0.13359823  0.00419892]
Fidelity lower bound: 0.9999996055385483
Fidelity witness on exp state: 0.8757471971277551


In [ ]:
np.shape(q1)

In [ ]:
2**8

In [ ]:
np.li

In [ ]:
np.linalg.norm(q4 - ham)

In [ ]:
import numpy as np
import cvxpy as cp

# Define Pauli matrices
I = np.eye(2)
sx = np.array([[0, 1], [1, 0]], dtype=complex)
sy = np.array([[0, -1j], [1j, 0]], dtype=complex)
sz = np.array([[1, 0], [0, -1]], dtype=complex)

# Tensor product utility
def kron(*args):
    result = np.array([[1]], dtype=complex)
    for op in args:
        result = np.kron(result, op)
    return result

# --- 2-site XXZ Hamiltonian: H = SxSx + SySy + ΔSzSz ---
Delta = 1.0  # XXZ anisotropy parameter

H = (
    kron(sx, sx) +
    kron(sy, sy) +
    Delta * kron(sz, sz)
)

# --- Second conserved quantity: total Sz ---
Q2 = kron(sz, I) + kron(I, sz)

# List of conserved charges
Q_list = [H, Q2]
Q_list = [Q / np.linalg.norm(Q) for Q in Q_list]

# --- Target state: singlet |ψ⟩ = (|01⟩ - |10⟩)/√2 ---
zero = np.array([1, 0], dtype=complex)
one = np.array([0, 1], dtype=complex)

psi_01 = np.kron(zero, one)
psi_10 = np.kron(one, zero)
psi_target = (psi_01 - psi_10) / np.sqrt(2)

# --- Expectation values in target state ---
q_target = np.array([np.vdot(psi_target, Q @ psi_target).real for Q in Q_list])

# --- Define CVXPY variables ---
alpha = cp.Variable(len(Q_list))

# Define W = sum_i alpha_i Q_i
W_expr = sum(alpha[i] * Q_list[i] for i in range(len(Q_list))) + 1e-5 * np.eye(4)


# --- Constraints ---
constraints = [
    W_expr >> 0,  # Positive semidefinite
    alpha @ q_target == 1  # Normalization on target state
]

# --- Solve the SDP ---
objective = cp.Maximize(0)  # Just solve feasibility
prob = cp.Problem(objective, constraints)
# prob.solve(
#     solver=cp.SCS,
#     eps=1e-6,           # Higher precision
#     verbose=True        # Shows solver progress
# )

prob.solve(solver=cp.CVXOPT)

print("Solver status:", prob.status)


# --- Construct final witness ---
W_opt = sum(alpha.value[i] * Q_list[i] for i in range(len(Q_list)))
W_opt_real = W_opt.real  # Strip small imaginary parts

# --- Fidelity witness on target state (should be 1) ---
fidelity_on_target = np.vdot(psi_target, W_opt @ psi_target).real

# --- Print results ---
print("Optimal α_i:", alpha.value)
print("Fidelity lower bound on target state:", fidelity_on_target)


Solver status: infeasible


TypeError: 'NoneType' object is not subscriptable

In [14]:
import numpy as np
import cvxpy as cp

# ... define Pauli matrices, kron, psi_target as before ...

# XXZ charges normalized
Q_list = [H / np.linalg.norm(H), Q2 / np.linalg.norm(Q2)]

# # Add the projector onto target state
# P = np.outer(psi_target, psi_target.conj())

# # Normalize projector
# P /= np.linalg.norm(P)

# Q_list.append(P)

# Compute expectation values of Q_i in psi_target
q_target = np.array([np.vdot(psi_target, Q @ psi_target).real for Q in Q_list])

# Variables: alpha coefficients and slack s
alpha = cp.Variable(len(Q_list))
s = cp.Variable(nonneg=True)

# Build witness
W_expr = sum(alpha[i] * Q_list[i] for i in range(len(Q_list)))

# Constraints: W PSD, approx normalization with slack s
constraints = [
    W_expr >> 0,
    alpha @ q_target + s == 1,
    s >= 0
]

# Objective: minimize slack s (make normalization tight)
prob = cp.Problem(cp.Minimize(s), constraints)

prob.solve(solver=cp.SCS, eps=1e-6, verbose=True)

print("Solver status:", prob.status)
print("Optimal slack s:", s.value)
print("alpha:", alpha.value)

if alpha.value is not None:
    W_opt = sum(alpha.value[i] * Q_list[i] for i in range(len(Q_list)))
    fidelity_witness_val = np.vdot(psi_target, W_opt @ psi_target).real
    print("Fidelity witness value on target state:", fidelity_witness_val)
else:
    print("No solution found.")


(CVXPY) Sep 17 12:33:52 PM: Your problem has 3 variables, 18 constraints, and 0 parameters.
(CVXPY) Sep 17 12:33:52 PM: It is compliant with the following grammars: DCP, DQCP
(CVXPY) Sep 17 12:33:52 PM: (If you need to solve this problem multiple times, but with different data, consider using parameters.)
(CVXPY) Sep 17 12:33:52 PM: CVXPY will first compile your problem; then, it will invoke a numerical solver to obtain a solution.
(CVXPY) Sep 17 12:33:52 PM: Your problem is compiled with the CPP canonicalization backend.
(CVXPY) Sep 17 12:33:52 PM: Compiling problem (target solver=SCS).
(CVXPY) Sep 17 12:33:52 PM: Reduction chain: Complex2Real -> Dcp2Cone -> CvxAttr2Constr -> ConeMatrixStuffing -> SCS
(CVXPY) Sep 17 12:33:52 PM: Applying reduction Complex2Real
(CVXPY) Sep 17 12:33:52 PM: Applying reduction Dcp2Cone
(CVXPY) Sep 17 12:33:52 PM: Applying reduction CvxAttr2Constr
(CVXPY) Sep 17 12:33:52 PM: Applying reduction ConeMatrixStuffing
(CVXPY) Sep 17 12:33:52 PM: Applying reducti

                                     CVXPY                                     
                                     v1.7.2                                    
-------------------------------------------------------------------------------
                                  Compilation                                  
-------------------------------------------------------------------------------
-------------------------------------------------------------------------------
                                Numerical solver                               
-------------------------------------------------------------------------------
------------------------------------------------------------------
	       SCS v3.2.8 - Splitting Conic Solver
	(c) Brendan O'Donoghue, Stanford University, 2012
------------------------------------------------------------------
problem:  variables n: 3, constraints m: 39
cones: 	  z: primal zero / dual free vars: 1
	  l: linear vars: 2
	  s: psd vars: 36, s

In [15]:
import numpy as np
import cvxpy as cp

# Define Q_list and psi_target as before
# (Q_list normalized if you want)

alpha = cp.Variable(len(Q_list))

W_expr = sum(alpha[i] * Q_list[i] for i in range(len(Q_list)))

# Constraint: PSD witness
constraints = [
    W_expr >> 0,
]

# Objective: maximize fidelity lower bound on target state
objective = cp.Maximize(alpha @ q_target)

prob = cp.Problem(objective, constraints)
prob.solve(solver=cp.SCS, eps=1e-6, verbose=True)

print("Solver status:", prob.status)
print("Optimal α:", alpha.value)
print("Fidelity lower bound:", alpha.value @ q_target if alpha.value is not None else None)


(CVXPY) Sep 17 12:35:15 PM: Your problem has 2 variables, 16 constraints, and 0 parameters.
(CVXPY) Sep 17 12:35:15 PM: It is compliant with the following grammars: DCP, DQCP
(CVXPY) Sep 17 12:35:15 PM: (If you need to solve this problem multiple times, but with different data, consider using parameters.)
(CVXPY) Sep 17 12:35:15 PM: CVXPY will first compile your problem; then, it will invoke a numerical solver to obtain a solution.
(CVXPY) Sep 17 12:35:15 PM: Your problem is compiled with the CPP canonicalization backend.
(CVXPY) Sep 17 12:35:15 PM: Compiling problem (target solver=SCS).
(CVXPY) Sep 17 12:35:15 PM: Reduction chain: Complex2Real -> FlipObjective -> Dcp2Cone -> CvxAttr2Constr -> ConeMatrixStuffing -> SCS
(CVXPY) Sep 17 12:35:15 PM: Applying reduction Complex2Real
(CVXPY) Sep 17 12:35:15 PM: Applying reduction FlipObjective
(CVXPY) Sep 17 12:35:15 PM: Applying reduction Dcp2Cone
(CVXPY) Sep 17 12:35:15 PM: Applying reduction CvxAttr2Constr
(CVXPY) Sep 17 12:35:15 PM: Appl

                                     CVXPY                                     
                                     v1.7.2                                    
-------------------------------------------------------------------------------
                                  Compilation                                  
-------------------------------------------------------------------------------
-------------------------------------------------------------------------------
                                Numerical solver                               
-------------------------------------------------------------------------------
------------------------------------------------------------------
	       SCS v3.2.8 - Splitting Conic Solver
	(c) Brendan O'Donoghue, Stanford University, 2012
------------------------------------------------------------------
problem:  variables n: 2, constraints m: 36
cones: 	  s: psd vars: 36, ssize: 1
settings: eps_abs: 1.0e-06, eps_rel: 1.0e-06, eps_i

In [31]:
import numpy as np
import cvxpy as cp

# ... your Q_list and q_target ...

# Add identity to charges
Q_list = [H / np.linalg.norm(H), Q2 / np.linalg.norm(Q2)]
q_target = np.array([np.vdot(psi_target, Q @ psi_target).real for Q in Q_list])
I = np.eye(Q_list[0].shape[0])
Q_list.append(I)
q_target = np.append(q_target, 1.0)

alpha = cp.Variable(len(Q_list))
W_expr = sum(alpha[i] * Q_list[i] for i in range(len(Q_list)))

lambda_reg = 1e-2  # Tune this value if needed

constraints = [
    W_expr >> 0,
    alpha @ q_target == 1   # normalize witness to have expectation 1 on target
]

objective = cp.Maximize(0)  # can also be any objective; here just maximize fidelity bound

#objective = cp.Maximize(alpha @ q_target - lambda_reg * cp.norm1(alpha))

prob = cp.Problem(objective, constraints)
prob.solve(solver=cp.SCS, eps=1e-5, verbose=True)

print("Status:", prob.status)
print("alpha:", alpha.value)
print("Fidelity lower bound:", alpha.value @ q_target if alpha.value is not None else None)



(CVXPY) Sep 17 12:59:05 PM: Your problem has 3 variables, 17 constraints, and 0 parameters.
(CVXPY) Sep 17 12:59:05 PM: It is compliant with the following grammars: DCP, DQCP
(CVXPY) Sep 17 12:59:05 PM: (If you need to solve this problem multiple times, but with different data, consider using parameters.)
(CVXPY) Sep 17 12:59:05 PM: CVXPY will first compile your problem; then, it will invoke a numerical solver to obtain a solution.
(CVXPY) Sep 17 12:59:05 PM: Your problem is compiled with the CPP canonicalization backend.
(CVXPY) Sep 17 12:59:05 PM: Compiling problem (target solver=SCS).
(CVXPY) Sep 17 12:59:05 PM: Reduction chain: Complex2Real -> FlipObjective -> Dcp2Cone -> CvxAttr2Constr -> ConeMatrixStuffing -> SCS
(CVXPY) Sep 17 12:59:05 PM: Applying reduction Complex2Real
(CVXPY) Sep 17 12:59:05 PM: Applying reduction FlipObjective
(CVXPY) Sep 17 12:59:05 PM: Applying reduction Dcp2Cone
(CVXPY) Sep 17 12:59:05 PM: Applying reduction CvxAttr2Constr
(CVXPY) Sep 17 12:59:05 PM: Appl

                                     CVXPY                                     
                                     v1.7.2                                    
-------------------------------------------------------------------------------
                                  Compilation                                  
-------------------------------------------------------------------------------
-------------------------------------------------------------------------------
                                Numerical solver                               
-------------------------------------------------------------------------------
------------------------------------------------------------------
	       SCS v3.2.8 - Splitting Conic Solver
	(c) Brendan O'Donoghue, Stanford University, 2012
------------------------------------------------------------------
problem:  variables n: 3, constraints m: 37
cones: 	  z: primal zero / dual free vars: 1
	  s: psd vars: 36, ssize: 1
settings: eps

In [47]:
import cvxpy as cp
import numpy as np

# --- Setup: system and charges ---
d = 8  # Hilbert space dimension
n_charges = 3

# Example conserved charges (random Hermitian for illustration)
Q_list = [np.random.randn(d, d) + 1j * np.random.randn(d, d) for _ in range(n_charges)]
Q_list = [0.5 * (Q + Q.conj().T) for Q in Q_list]  # Make Hermitian

# --- Target state |psi> ---
psi = np.random.randn(d) + 1j * np.random.randn(d)
psi /= np.linalg.norm(psi)

# --- Reference state rho ---
# You can use any state here, pure or mixed
phi = np.random.randn(d) + 1j * np.random.randn(d)
phi /= np.linalg.norm(phi)
rho = np.outer(phi, phi.conj())  # pure state for example

# --- Compute expectations in rho ---
q_rho = np.array([np.real(np.trace(rho @ Q)) for Q in Q_list])

# --- CVXPY optimization to lower-bound fidelity ---
sigma = cp.Variable((d, d), hermitian=True)
constraints = [
    sigma >> 0,
    cp.trace(sigma) == 1,
]

for i, Q in enumerate(Q_list):
    constraints.append(cp.real(cp.trace(sigma @ Q)) == q_rho[i])

# Objective: minimize overlap with target state
proj = np.outer(psi, psi.conj())
fidelity_bound = cp.real(cp.trace(sigma @ proj))

prob = cp.Problem(cp.Minimize(fidelity_bound), constraints)
prob.solve(solver=cp.SCS, eps=1e-7)

print("True lower bound on fidelity:", fidelity_bound.value)


True lower bound on fidelity: -7.033322030826242e-08


In [59]:
import numpy as np
import cvxpy as cp
from scipy.linalg import expm

# --- Define Pauli matrices ---
def pauli(op):
    if op == 'I':
        return np.eye(2)
    elif op == 'X':
        return np.array([[0, 1], [1, 0]])
    elif op == 'Y':
        return np.array([[0, -1j], [1j, 0]])
    elif op == 'Z':
        return np.array([[1, 0], [0, -1]])
    else:
        raise ValueError("Unknown operator")

def kron_n(ops):
    out = ops[0]
    for op in ops[1:]:
        out = np.kron(out, op)
    return out

# --- XXZ conserved charges: H, total Z, identity ---
def build_xxz_charges(L, delta=1.0):
    d = 2 ** L
    I = np.eye(2)
    charges = []

    # XXZ Hamiltonian
    H = np.zeros((d, d), dtype=complex)
    for i in range(L - 1):
        for term in ['X', 'Y', 'Z']:
            ops = [I for _ in range(L)]
            ops[i] = pauli(term)
            ops[i+1] = pauli(term)
            coef = 1.0 if term != 'Z' else delta
            H += coef * kron_n(ops)
    charges.append(H)

    # Total magnetization Z
    Z_total = np.zeros((d, d), dtype=complex)
    for i in range(L):
        ops = [I for _ in range(L)]
        ops[i] = pauli('Z')
        Z_total += kron_n(ops)
    charges.append(Z_total)

    # Identity
    charges.append(np.eye(d))

    return charges

# --- System setup ---
L = 4  # Chain length
d = 2 ** L
charges = build_xxz_charges(L)

# --- Target state: U |000> ---
zero = np.array([1, 0])
state_0 = zero
for _ in range(L - 1):
    state_0 = np.kron(state_0, zero)

# Random unitary
rng = np.random.default_rng(42)
U = expm(1j * rng.standard_normal((d, d)))
psi = U @ state_0
psi /= np.linalg.norm(psi)

# --- Reference state: slightly depolarized version of psi ---
theta = 0.2
rho = np.outer(psi, psi.conj())
rho = (1 - theta) * rho + theta * np.eye(d) / d  # mixed state

# --- Expectation values of charges in rho ---
q_rho = np.array([np.real(np.trace(rho @ Q)) for Q in charges])

# --- Fidelity lower bound via convex optimization ---
sigma = cp.Variable((d, d), hermitian=True)
constraints = [sigma >> 0, cp.trace(sigma) == 1]
for i, Q in enumerate(charges):
    constraints.append(cp.real(cp.trace(sigma @ Q)) == q_rho[i])

# Objective: minimize overlap with target
proj = np.outer(psi, psi.conj())
fidelity_bound = cp.real(cp.trace(sigma @ proj))

# Solve
prob = cp.Problem(cp.Minimize(fidelity_bound), constraints)
prob.solve(solver=cp.SCS, eps=1e-6)

# --- Report ---
print("Fidelity lower bound (squared):", fidelity_bound.value)
#print("Fidelity lower bound:", np.sqrt(fidelity_bound.value))


Fidelity lower bound (squared): 3.9641364503056627e-13


In [63]:
import numpy as np
import scipy.sparse as sp

def kron_N(ops):
    """Kronecker product of list of sparse operators."""
    result = ops[0]
    for op in ops[1:]:
        result = sp.kron(result, op, format='csr')
    return result

def local_pauli(pauli, site, N):
    """Construct local Pauli operator on site (0-based) in N spins."""
    I = sp.identity(2, format='csr')
    ops = [I] * N
    ops[site] = pauli
    return kron_N(ops)

def two_body_pauli(pauli1, site1, pauli2, site2, N):
    """Two-body operator acting on site1 and site2."""
    I = sp.identity(2, format='csr')
    ops = [I] * N
    ops[site1] = pauli1
    ops[site2] = pauli2
    return kron_N(ops)

# Pauli matrices as sparse:
sx = sp.csr_matrix(np.array([[0,1],[1,0]], dtype=complex))
sy = sp.csr_matrix(np.array([[0,-1j],[1j,0]], dtype=complex))
sz = sp.csr_matrix(np.array([[1,0],[0,-1]], dtype=complex))
id2 = sp.identity(2, format='csr')

N = 4  # system size
Delta = 0.5  # anisotropy parameter

# Identity operator
I_N = sp.identity(2**N, format='csr')

# Total magnetization Z_tot = sum_i Z_i
Z_list = [local_pauli(sz, i, N) for i in range(N)]
Z_tot = sum(Z_list)

# XXZ Hamiltonian H = sum_i (X_i X_{i+1} + Y_i Y_{i+1} + Delta * Z_i Z_{i+1})
H = sp.csr_matrix((2**N, 2**N), dtype=complex)
for i in range(N-1):
    H += (two_body_pauli(sx, i, sx, i+1, N)
          + two_body_pauli(sy, i, sy, i+1, N)
          + Delta * two_body_pauli(sz, i, sz, i+1, N))

# Add local Z_i and two-point Z_i Z_{i+1} charges
ZZ_list = [two_body_pauli(sz, i, sz, i+1, N) for i in range(N-1)]

# Compose full list of charges:
Q_list = [I_N, Z_tot, H] + Z_list + ZZ_list

print(f"Number of charges: {len(Q_list)}")

import numpy as np

q_list = []

for Q in Q_list:
    val = np.vdot(psi, Q @ psi)  # <psi|Q|psi>
    q_list.append(np.real_if_close(val))  # real expectation, numerical rounding

q_list = np.array(q_list)
print("Expectation values q_list:", q_list)




Number of charges: 10
Expectation values q_list: [ 1.          0.23932882 -0.34968904  0.13444812 -0.11172648  0.06791823
  0.14868894  0.00558768  0.10235763  0.03574029]


In [71]:
import cvxpy as cp

N = 4  # number of spins
dim = 2 ** N

# Build operators Q_list for this N (16x16 matrices)
# Build psi with length 16, e.g., the ground state or any vector of size 16

print(f"Dimension of operators: {dim}")
print(f"Length of psi: {len(psi)}")


d = 2**N
sigma = cp.Variable((d,d), hermitian=True)

constraints = [
    sigma >> 0,
    cp.trace(sigma) == 1,
]

for i, Q in enumerate(Q_list):
    constraints.append(cp.real(cp.trace(sigma @ Q.toarray())) == q_list[i])

F_lb = cp.real(cp.trace(sigma @ np.outer(psi, psi.conj())))
objective = cp.Minimize(F_lb)

prob = cp.Problem(objective, constraints)
prob.solve(solver=cp.SCS, eps=1e-10)

print("Fidelity lower bound:", prob.value)



Dimension of operators: 16
Length of psi: 16
Fidelity lower bound: -3.848420090966781e-12


In [72]:
sigma_opt = sigma.value  # optimal density matrix (numpy array)
expectation_after_opt = []

for Q in Q_list:
    val = np.trace(sigma_opt @ Q)
    expectation_after_opt.append(np.real_if_close(val))

expectation_after_opt = np.array(expectation_after_opt)

print("Expectation values on optimized state:")
print(expectation_after_opt)


Expectation values on optimized state:
[ 1.          0.23932882 -0.34968904  0.13444812 -0.11172648  0.06791823
  0.14868894  0.00558768  0.10235763  0.03574029]


In [73]:
q_list

array([ 1.        ,  0.23932882, -0.34968904,  0.13444812, -0.11172648,
        0.06791823,  0.14868894,  0.00558768,  0.10235763,  0.03574029])

In [141]:
import numpy as np

# Pauli matrices
I = np.eye(2, dtype=complex)
X = np.array([[0, 1], [1, 0]], dtype=complex)
Y = np.array([[0, -1j], [1j, 0]], dtype=complex)
Z = np.array([[1, 0], [0, -1]], dtype=complex)

def kron_N(ops):
    """Kronecker product of list of operators."""
    result = ops[0]
    for op in ops[1:]:
        result = np.kron(result, op)
    return result

def spin_op(site, op, L):
    """Operator acting as 'op' on site and identity elsewhere."""
    ops = [I] * L
    ops[site] = op
    return kron_N(ops)

def two_site_op(site1, site2, op1, op2, L):
    """Operator acting as op1 on site1 and op2 on site2."""
    ops = [I] * L
    ops[site1] = op1
    ops[site2] = op2
    return kron_N(ops)

def XXZ_Hamiltonian(L, Delta):
    """XXZ Hamiltonian for length L and anisotropy Delta."""
    H = np.zeros((2**L, 2**L), dtype=complex)
    for i in range(L-1):
        H += (two_site_op(i, i+1, X, X, L) +
              two_site_op(i, i+1, Y, Y, L) +
              Delta * two_site_op(i, i+1, Z, Z, L))
    H += (two_site_op(0, L-1, X, X, L) +
            two_site_op(0, L-1, Y, Y, L) +
            Delta * two_site_op(0, L-1, Z, Z, L))
    return H

def boost_operator(L):
    """Boost operator B for chain length L.
    B = sum_j j h_{j,j+1}
    """
    B = np.zeros((2**L, 2**L), dtype=complex)
    for j in range(L-1):
        # Local Hamiltonian term h_{j,j+1}
        h_j = (two_site_op(j, j+1, X, X, L) +
               two_site_op(j, j+1, Y, Y, L) +
               Delta * two_site_op(j, j+1, Z, Z, L))
        B += j * h_j

    h_j = (two_site_op(L-1, 0, X, X, L) +
           two_site_op(L-1, 0, Y, Y, L) +
           Delta * two_site_op(L-1, 0, Z, Z, L))
    B += (L-1) * h_j  # periodic boundary
    return 1j*B

def operator_inner_product(A, B):
    """Normalized Hilbert-Schmidt inner product."""
    return np.trace(np.conjugate(A.T) @ B).real / A.shape[0]

def fold_charge(n, tilde_Qs, folded_Qs, Delta, p):
    """Fold the charge Q_n recursively."""
    Qn = tilde_Qs[n].copy()
    for k in range(1, n):
        c = operator_inner_product(Qn, folded_Qs[k]) / operator_inner_product(folded_Qs[k], folded_Qs[k])
        Qn = Qn - (Delta**(p[n] - p[k])) * c * folded_Qs[k]
    Qn = Qn / (Delta**p[n])
    return Qn

# Parameters
L = 5
Delta = 0.2  # large anisotropy parameter
p = {1: 1, 2: 2, 3: 3}  # scaling exponents for charges (assumed)

B = boost_operator(L)
xxz_ham = XXZ_Hamiltonian(L, Delta)

# Compute tilde_Q1 (total magnetization along z)
tilde_Q1 = xxz_ham #sum(spin_op(i, Z, L) for i in range(L))

# Compute tilde_Q2 = [B, tilde_Q1]

tilde_Q2 = B @ tilde_Q1 - tilde_Q1 @ B

# Generate tilde_Q3 = [B, tilde_Q2]
tilde_Q3 = B @ tilde_Q2 - tilde_Q2 @ B

# Store tilde charges
tilde_Qs = {1: tilde_Q1, 2: tilde_Q2, 3: tilde_Q3}

# # Folded charges dictionary
# folded_Qs = {}

# for n in range(1, 4):
#     folded_Qs[n] = fold_charge(n, tilde_Qs, folded_Qs, Delta, p)

# # Compare norms (for sanity check)
# print("Norms of folded charges Q_n:")
# for n in folded_Qs:
#     norm = np.linalg.norm(folded_Qs[n])
#     print(f"||Q_{n}|| = {norm:.4f}")

# You can also print the folded charges or their matrix entries for detailed comparison.


In [142]:
def commutator(A, B):
    return A @ B - B @ A

print("\nChecking commutation of folded charges with folded Hamiltonian (Q_2):\n")

for n in [1, 2, 3]:
    comm = commutator(tilde_Qs[n], xxz_ham)
    comm_norm = np.linalg.norm(comm)
    print(f"||[Q_{n}, H_xxz]|| = {comm_norm:.2e} (should be ~0)")



Checking commutation of folded charges with folded Hamiltonian (Q_2):

||[Q_1, H_xxz]|| = 0.00e+00 (should be ~0)
||[Q_2, H_xxz]|| = 3.45e+02 (should be ~0)
||[Q_3, H_xxz]|| = 4.06e+03 (should be ~0)


In [140]:
np.linalg.norm(tilde_Q2)

np.float64(0.0)

In [22]:
import numpy as np

# Pauli matrices
sx = np.array([[0,1],[1,0]], dtype=complex)
sy = np.array([[0,-1j],[1j,0]], dtype=complex)
sz = np.array([[1,0],[0,-1]], dtype=complex)
id2 = np.eye(2, dtype=complex)

def kron_n(ops):
    result = ops[0]
    for op in ops[1:]:
        result = np.kron(result, op)
    return result

def local_hamiltonian(Delta):
    return np.kron(sx, sx) + np.kron(sy, sy) + Delta * np.kron(sz, sz)

def embed_two_site_op(op, j, L):
    jp1 = (j + 1) % L
    left_count = min(j, jp1)
    right_count = L - left_count - 2

    left_id = np.eye(2**left_count, dtype=complex) if left_count > 0 else np.array([[1]], dtype=complex)
    right_id = np.eye(2**right_count, dtype=complex) if right_count > 0 else np.array([[1]], dtype=complex)

    full_op = np.kron(np.kron(left_id, op), right_id)
    return full_op

def total_Sz(L):
    Sz_tot = np.zeros((2**L, 2**L), dtype=complex)
    for j in range(L):
        ops = [id2] * L
        ops[j] = sz
        Sz_tot += kron_n(ops)
    return Sz_tot

def boost_operator(h_local, L):
    B = np.zeros((2**L, 2**L), dtype=complex)
    for j in range(L):
        hj = embed_two_site_op(h_local, j, L)
        B += j * hj
    return B

def commutator(A, B):
    return A @ B - B @ A

def boost_operator_centered(h_local, L):
    B = np.zeros((2**L, 2**L), dtype=complex)
    center = (L + 1) / 2
    for j in range(L):
        hj = embed_two_site_op(h_local, j, L)
        B += (j + 1 - center) * hj  # j+1 to index from 1 instead of 0
    return B


def generate_charges(L, Delta, n_charges=4):
    h_local = local_hamiltonian(Delta)
    Sz_tot = total_Sz(L)

    Q_list = [Sz_tot]

    H = np.zeros((2**L, 2**L), dtype=complex)
    for j in range(L):
        H += embed_two_site_op(h_local, j, L)
    Q_list.append(H)

    B = boost_operator_centered(h_local, L)

    for n in range(2, n_charges):
        Q_next = commutator(B, Q_list[-1])
        Q_list.append(Q_next)

    return Q_list

# Example:
L = 3
Delta = 1.128
charges = generate_charges(L, Delta, n_charges=4)

for i, Q in enumerate(charges, 1):
    print(f"Q{i} norm: {np.linalg.norm(Q):.3e}")

# Check commutation:
for i in range(len(charges)):
    for j in range(i+1, len(charges)):
        comm = commutator(charges[i], charges[j])
        print(f"Norm([Q{i+1}, Q{j+1}]) = {np.linalg.norm(comm):.3e}")









Q1 norm: 4.899e+00
Q2 norm: 1.144e+01
Q3 norm: 0.000e+00
Q4 norm: 0.000e+00
Norm([Q1, Q2]) = 0.000e+00
Norm([Q1, Q3]) = 0.000e+00
Norm([Q1, Q4]) = 0.000e+00
Norm([Q2, Q3]) = 0.000e+00
Norm([Q2, Q4]) = 0.000e+00
Norm([Q3, Q4]) = 0.000e+00


In [21]:
L = 3
Delta = 1.128
charges = generate_charges(L, Delta, n_charges=4)

for i, Q in enumerate(charges, 1):
    print(f"Q{i} operator norm: {np.linalg.norm(Q):.3e}")

# Check commutation between charges
for i in range(len(charges)):
    for j in range(i+1, len(charges)):
        comm = commutator(charges[i], charges[j])
        norm_comm = np.linalg.norm(comm)
        print(f"Norm([Q{i+1}, Q{j+1}]) = {norm_comm:.3e}")


Q1 operator norm: 4.899e+00
Q2 operator norm: 1.144e+01
Q3 operator norm: 0.000e+00
Q4 operator norm: 0.000e+00
Norm([Q1, Q2]) = 0.000e+00
Norm([Q1, Q3]) = 0.000e+00
Norm([Q1, Q4]) = 0.000e+00
Norm([Q2, Q3]) = 0.000e+00
Norm([Q2, Q4]) = 0.000e+00
Norm([Q3, Q4]) = 0.000e+00
